# 공간 오디오 공식 베이스라인

2초 stereo WAV에서 음원 종류와 azimuth를 함께 예측합니다. 마지막 셀에서
Kaggle 제출 파일을 생성합니다.


In [ ]:
import math
import os
import random
import shutil
import wave
import zipfile
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 17
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "0") == "1"
DATASET_SLUG = 'single-source-spatial-audio'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

WORK = Path(os.environ.get("KAGGLE_WORK_DIR", "/kaggle/working"))
if not WORK.parent.exists():
    WORK = Path("artifacts/kaggle_notebook_run").resolve()
WORK.mkdir(parents=True, exist_ok=True)

def find_data_root():
    override = os.environ.get("KAGGLE_DATA_DIR")
    search_root = Path(override) if override else Path("/kaggle/input")
    direct = search_root / DATASET_SLUG
    candidates = [direct, search_root] + [p.parent for p in search_root.rglob("train.csv")]
    valid = []
    for candidate in candidates:
        if all((candidate / name).is_file() for name in (
            "train.csv", "test.csv", "sample_submission.csv", "class_map.csv"
        )):
            valid.append(candidate.resolve())
    valid = list(dict.fromkeys(valid))
    if len(valid) != 1:
        raise RuntimeError(f"데이터 디렉터리를 하나로 결정할 수 없습니다: {valid}")
    return valid[0]

DATA = find_data_root()
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
class_map = pd.read_csv(DATA / "class_map.csv").sort_values("class_id")

assert train.columns.tolist() == ["id", "filename", "sound_class", "azimuth"]
assert test.columns.tolist() == ["id", "filename"]
assert sample.columns.tolist() == ["id", "sound_class", "azimuth"]
assert not train.isna().any().any() and not test.isna().any().any()
assert train["id"].is_unique and test["id"].is_unique
assert test["id"].astype(str).tolist() == sample["id"].astype(str).tolist()

SOUND_CLASSES = tuple(class_map["sound_class"].astype(str))
AZIMUTHS = tuple(sorted(train["azimuth"].astype(int).unique().tolist()))
assert set(train["sound_class"].astype(str)) == set(SOUND_CLASSES)
assert set(train["azimuth"].astype(int)) == set(AZIMUTHS)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("data:", DATA)
print("train/test:", len(train), len(test), "device:", DEVICE)


## 데이터와 특징

공식 오디오 ZIP은 WAV가 루트에 바로 있는 평면 구조입니다. 아래 셀은
Kaggle에서 ZIP 또는 디렉터리로 보이는 경우를 모두 자동으로 찾습니다.
train은 class×azimuth 기준으로 나누고 L/R log-mel, ILD, cos(IPD),
sin(IPD)의 5채널 특징을 사용합니다.


In [ ]:
def _audio_directory_is_complete(directory, required):
    return directory.is_dir() and all(
        (directory / filename).is_file() for filename in required
    )

def prepare_audio_dir(split, filenames):
    original_names = [str(name) for name in filenames]
    required = tuple(dict.fromkeys(Path(name).name for name in original_names))
    if len(required) != len(original_names):
        raise ValueError(f"{split} audio filenames must be unique")
    if any(name != basename for name, basename in zip(original_names, required)):
        raise ValueError(f"{split} audio filenames must not contain directories")

    direct = DATA / f"{split}_audio"
    # Kaggle may expose a directory upload as either `train_audio/*.wav` or
    # `train_audio/train_audio/*.wav`.  Only accept a directory after checking
    # that every filename needed by this run is directly inside it.
    directory_candidates = [direct, direct / direct.name]
    if required:
        directory_candidates.extend(
            match.parent for match in DATA.rglob(required[0]) if match.is_file()
        )
    checked = set()
    for candidate in directory_candidates:
        resolved = candidate.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        if _audio_directory_is_complete(candidate, required):
            print(f"{split} audio directory:", candidate)
            return candidate, None

    archives = [DATA / f"{split}_audio.zip"]
    archives.extend(DATA.rglob(f"{split}_audio.zip"))
    archives = list(dict.fromkeys(path.resolve() for path in archives if path.is_file()))
    if len(archives) != 1:
        raise FileNotFoundError(
            f"{split}_audio 디렉터리 또는 ZIP을 하나 찾지 못했습니다: {archives}"
        )

    kaggle_temp = Path("/kaggle/temp")
    cache_parent = kaggle_temp if kaggle_temp.is_dir() else WORK / "_audio_cache"
    target = cache_parent / f"{split}_audio"
    target.mkdir(parents=True, exist_ok=True)
    required_set = set(required)
    with zipfile.ZipFile(archives[0]) as archive:
        # The published archives are flat (`train_*.wav` / `test_*.wav` at
        # the ZIP root).  Matching the validated basename also keeps older
        # archives with a `train_audio/` or `test_audio/` prefix usable.
        members = {}
        for member in archive.infolist():
            if member.is_dir():
                continue
            basename = Path(member.filename).name
            if basename in required_set:
                if basename in members:
                    raise RuntimeError(f"ZIP에 중복 파일명이 있습니다: {basename}")
                members[basename] = member
        missing = required_set - set(members)
        if missing:
            raise FileNotFoundError(f"ZIP에 WAV가 없습니다: {sorted(missing)[:5]}")
        for filename in sorted(required_set):
            output = target / filename
            if output.is_file():
                continue
            with archive.open(members[filename]) as source, output.open("wb") as destination:
                shutil.copyfileobj(source, destination)
    return target, target

def stratified_split(frame):
    rng = np.random.default_rng(SEED)
    fit_indices, validation_indices = [], []
    for _, indices in frame.groupby(["sound_class", "azimuth"], sort=True).groups.items():
        indices = np.asarray(list(indices), dtype=np.int64)
        rng.shuffle(indices)
        validation_count = 1 if FAST_DEV_RUN else max(1, int(round(0.20 * len(indices))))
        validation_indices.extend(indices[:validation_count])
        remaining = indices[validation_count:]
        fit_indices.extend(remaining[:2] if FAST_DEV_RUN else remaining)
    fit = frame.loc[fit_indices].sort_values("id").reset_index(drop=True)
    validation = frame.loc[validation_indices].sort_values("id").reset_index(drop=True)
    if fit.empty or validation.empty:
        raise ValueError("학습/검증 split이 비었습니다.")
    assert set(fit["id"]).isdisjoint(validation["id"])
    return fit, validation

fit, validation = stratified_split(train)
test_run = test.head(min(200, len(test))).copy() if FAST_DEV_RUN else test.copy()
train_audio, extracted_train = prepare_audio_dir("train", train["filename"])
test_audio, extracted_test = prepare_audio_dir("test", test_run["filename"])
print("fit/validation/inference:", len(fit), len(validation), len(test_run))


In [ ]:
DEFAULT_FEATURE_CONFIG = {'sample_rate': 16000, 'clip_seconds': 2.0, 'n_fft': 512, 'win_length': 400, 'hop_length': 160, 'n_mels': 64, 'f_min': 50.0, 'f_max': 8000.0, 'center': True, 'normalization': 'fixed', 'dynamic_range_db': 80.0, 'ild_clip_db': 30.0, 'eps': 1e-10}

_WINDOW_CACHE = {}

_MEL_CACHE = {}

def _pcm_bytes_to_float(raw: bytes, sample_width: int, channels: int) -> np.ndarray:
    if sample_width == 1:
        values = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sample_width == 2:
        values = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    elif sample_width == 3:
        packed = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3)
        values_i32 = (
            packed[:, 0].astype(np.int32)
            | (packed[:, 1].astype(np.int32) << 8)
            | (packed[:, 2].astype(np.int32) << 16)
        )
        values_i32 = np.where(values_i32 & 0x800000, values_i32 - 0x1000000, values_i32)
        values = values_i32.astype(np.float32) / 8388608.0
    elif sample_width == 4:
        values = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0
    else:
        raise ValueError(f"Unsupported PCM sample width: {sample_width} bytes")
    if values.size % channels:
        raise ValueError("Malformed WAV: sample count is not divisible by channel count")
    return values.reshape(-1, channels).T.copy()


def load_audio(path: str | Path) -> tuple[torch.Tensor, int]:
    """Load an integer PCM WAV as a float tensor shaped ``[channels, samples]``.

    Competition audio is PCM WAV, so the standard-library reader keeps this
    module independent of optional audio backends.  ``torchaudio`` is attempted
    only for uncommon WAV encodings that ``wave`` cannot decode.
    """

    audio_path = Path(path)
    try:
        with wave.open(str(audio_path), "rb") as wav_file:
            channels = wav_file.getnchannels()
            sample_rate = wav_file.getframerate()
            sample_width = wav_file.getsampwidth()
            frames = wav_file.getnframes()
            raw = wav_file.readframes(frames)
        array = _pcm_bytes_to_float(raw, sample_width, channels)
        return torch.from_numpy(array), int(sample_rate)
    except (wave.Error, ValueError, EOFError) as wave_error:
        try:
            import torchaudio

            waveform, sample_rate = torchaudio.load(str(audio_path))
            return waveform.to(torch.float32), int(sample_rate)
        except Exception as backend_error:  # pragma: no cover - backend-dependent
            raise RuntimeError(
                f"Could not decode {audio_path} as PCM WAV ({wave_error}); "
                f"torchaudio fallback also failed ({backend_error})"
            ) from backend_error


def _resample_linear(waveform: torch.Tensor, old_rate: int, new_rate: int) -> torch.Tensor:
    if old_rate == new_rate:
        return waveform
    new_length = int(round(waveform.shape[-1] * new_rate / old_rate))
    return F.interpolate(
        waveform.unsqueeze(0), size=new_length, mode="linear", align_corners=False
    ).squeeze(0)


def prepare_waveform(
    waveform: torch.Tensor,
    sample_rate: int,
    *,
    target_sample_rate: int = 16_000,
    target_samples: int = 32_000,
    random_crop: bool = False,
    strict_stereo: bool = True,
) -> torch.Tensor:
    """Validate/resample and crop or zero-pad audio without per-ear gain changes."""

    if waveform.ndim != 2:
        raise ValueError(f"Expected [channels, samples], got {tuple(waveform.shape)}")
    waveform = waveform.to(torch.float32)
    if not torch.isfinite(waveform).all():
        raise ValueError("Audio contains NaN or Inf")
    if waveform.shape[0] != 2:
        if strict_stereo:
            raise ValueError(f"Expected stereo audio, found {waveform.shape[0]} channel(s)")
        if waveform.shape[0] == 1:
            waveform = waveform.repeat(2, 1)
        else:
            waveform = waveform[:2]
    waveform = _resample_linear(waveform, int(sample_rate), int(target_sample_rate))

    difference = waveform.shape[-1] - int(target_samples)
    if difference > 0:
        start = random.randint(0, difference) if random_crop else difference // 2
        waveform = waveform[:, start : start + target_samples]
    elif difference < 0:
        pad = -difference
        if random_crop:
            left_pad = random.randint(0, pad)
        else:
            left_pad = pad // 2
        waveform = F.pad(waveform, (left_pad, pad - left_pad))
    return waveform.contiguous()


def _hz_to_mel(hz: torch.Tensor) -> torch.Tensor:
    return 2595.0 * torch.log10(1.0 + hz / 700.0)


def _mel_to_hz(mel: torch.Tensor) -> torch.Tensor:
    return 700.0 * (torch.pow(10.0, mel / 2595.0) - 1.0)


def _mel_filterbank(
    sample_rate: int,
    n_fft: int,
    n_mels: int,
    f_min: float,
    f_max: float,
    dtype: torch.dtype,
) -> torch.Tensor:
    key = (sample_rate, n_fft, n_mels, f_min, f_max, dtype)
    if key in _MEL_CACHE:
        return _MEL_CACHE[key]

    fft_frequencies = torch.linspace(0.0, sample_rate / 2, n_fft // 2 + 1, dtype=dtype)
    mel_min = _hz_to_mel(torch.tensor(float(f_min), dtype=dtype))
    mel_max = _hz_to_mel(torch.tensor(float(f_max), dtype=dtype))
    edges_hz = _mel_to_hz(torch.linspace(mel_min, mel_max, n_mels + 2, dtype=dtype))

    lower = edges_hz[:-2, None]
    center = edges_hz[1:-1, None]
    upper = edges_hz[2:, None]
    up_slope = (fft_frequencies[None, :] - lower) / (center - lower).clamp_min(1.0e-12)
    down_slope = (upper - fft_frequencies[None, :]) / (upper - center).clamp_min(1.0e-12)
    filters = torch.minimum(up_slope, down_slope).clamp_min(0.0)
    # Unit-sum filters make each mel value a local band average.
    filters = filters / filters.sum(dim=1, keepdim=True).clamp_min(1.0e-12)
    _MEL_CACHE[key] = filters
    return filters


def extract_spatial_features(
    waveform: torch.Tensor,
    sample_rate: int = 16_000,
    feature_config: Mapping[str, Any] | None = None,
) -> torch.Tensor:
    """Extract five stereo STFT features, returning ``[5, mel, time]``."""

    cfg = dict(DEFAULT_FEATURE_CONFIG)
    if feature_config:
        cfg.update(feature_config)
    target_rate = int(cfg["sample_rate"])
    target_samples = int(round(target_rate * float(cfg["clip_seconds"])))
    waveform = prepare_waveform(
        waveform,
        sample_rate,
        target_sample_rate=target_rate,
        target_samples=target_samples,
        random_crop=False,
        strict_stereo=True,
    )

    n_fft = int(cfg["n_fft"])
    win_length = int(cfg["win_length"])
    hop_length = int(cfg["hop_length"])
    window_key = (win_length, waveform.dtype)
    if window_key not in _WINDOW_CACHE:
        _WINDOW_CACHE[window_key] = torch.hann_window(win_length, dtype=waveform.dtype)
    window = _WINDOW_CACHE[window_key].to(waveform.device)
    spectra = torch.stft(
        waveform,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=bool(cfg.get("center", True)),
        pad_mode="reflect",
        return_complex=True,
    )

    mel_filters = _mel_filterbank(
        target_rate,
        n_fft,
        int(cfg["n_mels"]),
        float(cfg["f_min"]),
        float(cfg["f_max"]),
        waveform.dtype,
    ).to(waveform.device)
    power = spectra.abs().square()
    mel_power = torch.einsum("mf,cft->cmt", mel_filters, power)
    eps = float(cfg.get("eps", 1.0e-10))
    scale = 10.0 / math.log(10.0)
    mel_db = scale * torch.log(mel_power.clamp_min(eps))

    cross_spectrum = spectra[0] * spectra[1].conj()
    cross_real = torch.einsum("mf,ft->mt", mel_filters, cross_spectrum.real)
    cross_imag = torch.einsum("mf,ft->mt", mel_filters, cross_spectrum.imag)
    cross_norm = torch.sqrt(cross_real.square() + cross_imag.square()).clamp_min(eps)
    cos_ipd = cross_real / cross_norm
    sin_ipd = cross_imag / cross_norm
    ild_db = mel_db[0] - mel_db[1]

    normalization = str(cfg.get("normalization", "fixed")).lower()
    if normalization == "fixed":
        # One common reference is essential: separate references would erase ILD.
        if mel_power.max().item() > eps * 10:
            common_reference = mel_db.max()
        else:
            common_reference = mel_db.new_tensor(0.0)
        dynamic_range = float(cfg.get("dynamic_range_db", 80.0))
        relative_db = (mel_db - common_reference).clamp(-dynamic_range, 0.0)
        log_mel = 2.0 * (relative_db + dynamic_range) / dynamic_range - 1.0
        ild_clip = float(cfg.get("ild_clip_db", 30.0))
        ild = ild_db.clamp(-ild_clip, ild_clip) / ild_clip
    elif normalization in {"none", "db"}:
        log_mel = mel_db
        ild = ild_db
    else:
        raise ValueError(
            f"Unknown feature normalization {normalization!r}; expected 'fixed' or 'none'"
        )

    features = torch.stack((log_mel[0], log_mel[1], ild, cos_ipd, sin_ipd), dim=0)
    return torch.nan_to_num(features, nan=0.0, posinf=1.0, neginf=-1.0).to(torch.float32)


def apply_time_frequency_mask(
    features: torch.Tensor,
    *,
    max_time_frames: int = 20,
    max_frequency_bins: int = 8,
    probability: float = 0.5,
) -> torch.Tensor:
    """Apply one shared time mask and one shared frequency mask to all channels."""

    result = features.clone()
    if max_time_frames > 0 and random.random() < probability:
        width = random.randint(1, min(max_time_frames, result.shape[-1]))
        start = random.randint(0, result.shape[-1] - width)
        result[..., start : start + width] = 0.0
    if max_frequency_bins > 0 and random.random() < probability:
        width = random.randint(1, min(max_frequency_bins, result.shape[-2]))
        start = random.randint(0, result.shape[-2] - width)
        result[:, start : start + width, :] = 0.0
    return result


class SpatialAudioDataset(Dataset[dict[str, Any]]):
    """CSV-backed dataset for both labeled training and unlabeled inference."""

    def __init__(
        self,
        manifest: str | Path | pd.DataFrame,
        *,
        audio_dir: str | Path | None = None,
        class_names: Sequence[str] = SOUND_CLASSES,
        azimuths: Sequence[int] = AZIMUTHS,
        feature_config: Mapping[str, Any] | None = None,
        augment: bool = False,
        strict_stereo: bool = True,
        mask_probability: float = 0.5,
        max_time_mask: int = 20,
        max_frequency_mask: int = 8,
    ) -> None:
        if isinstance(manifest, pd.DataFrame):
            self.manifest = manifest.reset_index(drop=True).copy()
            self.manifest_path: Path | None = None
        else:
            self.manifest_path = Path(manifest)
            self.manifest = pd.read_csv(self.manifest_path)
        required = {"id", "filename"}
        missing = required.difference(self.manifest.columns)
        if missing:
            raise ValueError(f"Manifest is missing columns: {sorted(missing)}")

        self.audio_dir = Path(audio_dir) if audio_dir is not None else None
        self.class_names = tuple(str(name) for name in class_names)
        self.azimuths = tuple(int(angle) for angle in azimuths)
        self.class_to_index = {name: index for index, name in enumerate(self.class_names)}
        self.azimuth_to_index = {angle: index for index, angle in enumerate(self.azimuths)}
        self.feature_config = dict(DEFAULT_FEATURE_CONFIG)
        if feature_config:
            self.feature_config.update(feature_config)
        self.augment = bool(augment)
        self.strict_stereo = bool(strict_stereo)
        self.mask_probability = float(mask_probability)
        self.max_time_mask = int(max_time_mask)
        self.max_frequency_mask = int(max_frequency_mask)
        self.has_targets = {"sound_class", "azimuth"}.issubset(self.manifest.columns)

        if self.has_targets:
            unknown_classes = set(self.manifest["sound_class"].astype(str)) - set(self.class_names)
            if unknown_classes:
                raise ValueError(f"Unknown sound classes: {sorted(unknown_classes)}")
            unknown_angles = set(self.manifest["azimuth"].astype(int)) - set(self.azimuths)
            if unknown_angles:
                raise ValueError(f"Unknown azimuth values: {sorted(unknown_angles)}")

    def __len__(self) -> int:
        return len(self.manifest)

    def _audio_path(self, filename: str) -> Path:
        candidate = Path(filename)
        if candidate.is_absolute():
            return candidate
        if self.audio_dir is not None:
            return self.audio_dir / candidate
        if self.manifest_path is not None:
            return self.manifest_path.parent / candidate
        return candidate

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.manifest.iloc[index]
        audio_path = self._audio_path(str(row["filename"]))
        waveform, sample_rate = load_audio(audio_path)
        target_rate = int(self.feature_config["sample_rate"])
        target_samples = int(
            round(target_rate * float(self.feature_config["clip_seconds"]))
        )
        waveform = prepare_waveform(
            waveform,
            sample_rate,
            target_sample_rate=target_rate,
            target_samples=target_samples,
            random_crop=self.augment,
            strict_stereo=self.strict_stereo,
        )
        features = extract_spatial_features(waveform, target_rate, self.feature_config)
        if self.augment:
            features = apply_time_frequency_mask(
                features,
                max_time_frames=self.max_time_mask,
                max_frequency_bins=self.max_frequency_mask,
                probability=self.mask_probability,
            )

        item: dict[str, Any] = {
            "features": features,
            "id": str(row["id"]),
            "filename": str(row["filename"]),
        }
        if self.has_targets:
            sound_class = str(row["sound_class"])
            azimuth = int(row["azimuth"])
            item.update(
                {
                    "class_idx": self.class_to_index[sound_class],
                    "azimuth_idx": self.azimuth_to_index[azimuth],
                    "sound_class": sound_class,
                    "azimuth": azimuth,
                }
            )
        return item


DEFAULT_CHANNELS = (32, 48, 48, 64, 64, 96, 96, 128, 160)

DEFAULT_STRIDES = ((1, 1), (1, 1), (2, 2), (1, 1), (2, 2), (1, 1), (2, 2), (1, 1))

class ConvNormActivation(nn.Sequential):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        *,
        kernel_size: int = 3,
        stride: tuple[int, int] = (1, 1),
        groups: int = 1,
    ) -> None:
        padding = kernel_size // 2
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride=stride,
                padding=padding,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )


class DepthwiseSeparableBlock(nn.Module):
    """3x3 depthwise convolution followed by a 1x1 pointwise convolution."""

    def __init__(
        self, in_channels: int, out_channels: int, stride: tuple[int, int] = (1, 1)
    ) -> None:
        super().__init__()
        self.depthwise = ConvNormActivation(
            in_channels,
            in_channels,
            kernel_size=3,
            stride=stride,
            groups=in_channels,
        )
        self.pointwise = ConvNormActivation(
            in_channels, out_channels, kernel_size=1, stride=(1, 1)
        )
        self.use_residual = stride == (1, 1) and in_channels == out_channels

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        outputs = self.pointwise(self.depthwise(inputs))
        if self.use_residual:
            outputs = outputs + inputs
        return outputs


class SpatialBaselineCNN(nn.Module):
    """Official network-only baseline, designed for roughly 31 MMAC.

    With the default ``[1, 5, 64, 201]`` feature input the convolutions and two
    linear heads stay comfortably below the 100 MMAC competition limit.
    """

    def __init__(
        self,
        *,
        input_channels: int = 5,
        num_classes: int = 8,
        num_azimuths: int = 25,
        channels: Sequence[int] = DEFAULT_CHANNELS,
        dropout: float = 0.15,
    ) -> None:
        super().__init__()
        channels = tuple(int(channel) for channel in channels)
        if len(channels) != len(DEFAULT_STRIDES) + 1:
            raise ValueError(
                f"channels must contain {len(DEFAULT_STRIDES) + 1} values, got {len(channels)}"
            )
        self.input_channels = int(input_channels)
        self.num_classes = int(num_classes)
        self.num_azimuths = int(num_azimuths)
        self.channels = channels
        self.dropout_probability = float(dropout)

        # Early 2x downsampling plus two full-resolution separable blocks puts
        # useful capacity where spatial spectrogram detail is still available.
        self.stem = ConvNormActivation(
            self.input_channels, channels[0], kernel_size=3, stride=(2, 2)
        )
        self.blocks = nn.Sequential(
            *[
                DepthwiseSeparableBlock(in_ch, out_ch, stride)
                for in_ch, out_ch, stride in zip(channels[:-1], channels[1:], DEFAULT_STRIDES)
            ]
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(self.dropout_probability)
        self.sound_head = nn.Linear(channels[-1], self.num_classes)
        self.azimuth_head = nn.Linear(channels[-1], self.num_azimuths)

        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.01)
                nn.init.zeros_(module.bias)

    def model_config(self) -> dict[str, object]:
        return {
            "input_channels": self.input_channels,
            "num_classes": self.num_classes,
            "num_azimuths": self.num_azimuths,
            "channels": list(self.channels),
            "dropout": self.dropout_probability,
        }

    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        if inputs.ndim != 4 or inputs.shape[1] != self.input_channels:
            raise ValueError(
                f"Expected [batch, {self.input_channels}, mel, time], got {tuple(inputs.shape)}"
            )
        embedding = self.pool(self.blocks(self.stem(inputs))).flatten(1)
        embedding = self.dropout(embedding)
        return self.sound_head(embedding), self.azimuth_head(embedding)


In [ ]:
FEATURE_CONFIG = {
    "sample_rate": 16000,
    "clip_seconds": 2.0,
    "n_fft": 512,
    "win_length": 400,
    "hop_length": 160,
    "n_mels": 64,
    "f_min": 50.0,
    "f_max": 7800.0,
    "normalization": "fixed",
    "dynamic_range_db": 80.0,
    "ild_clip_db": 30.0,
    "eps": 1.0e-6,
}
NUM_WORKERS = int(os.environ.get("NOTEBOOK_NUM_WORKERS", "0" if FAST_DEV_RUN else "2"))
BATCH_SIZE = 32

train_dataset = SpatialAudioDataset(
    fit, audio_dir=train_audio, class_names=SOUND_CLASSES, azimuths=AZIMUTHS,
    feature_config=FEATURE_CONFIG, augment=True,
)
validation_dataset = SpatialAudioDataset(
    validation, audio_dir=train_audio, class_names=SOUND_CLASSES, azimuths=AZIMUTHS,
    feature_config=FEATURE_CONFIG, augment=False,
)
test_dataset = SpatialAudioDataset(
    test_run, audio_dir=test_audio, class_names=SOUND_CLASSES, azimuths=AZIMUTHS,
    feature_config=FEATURE_CONFIG, augment=False,
)

generator = torch.Generator().manual_seed(SEED)
loader_options = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0,
)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_options)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)


## 학습

작은 depthwise-separable CNN의 두 출력 head를 함께 학습하고 validation
Joint Macro-F1이 가장 높은 checkpoint를 저장합니다.


In [ ]:
def joint_macro_f1(true_class, true_azimuth, pred_class, pred_azimuth):
    true_class = np.asarray(true_class)
    pred_class = np.asarray(pred_class)
    true_azimuth = np.asarray(true_azimuth, dtype=float)
    pred_azimuth = np.asarray(pred_azimuth, dtype=float)
    tolerance_scores = []
    for tolerance in (5.0, 10.0, 20.0):
        matched = (true_class == pred_class) & (
            np.abs(true_azimuth - pred_azimuth) <= tolerance
        )
        class_scores = []
        for label in SOUND_CLASSES:
            true_positive = int(np.sum(matched & (true_class == label)))
            false_negative = int(np.sum(true_class == label)) - true_positive
            false_positive = int(np.sum(pred_class == label)) - true_positive
            denominator = 2 * true_positive + false_positive + false_negative
            class_scores.append(2 * true_positive / denominator if denominator else 0.0)
        tolerance_scores.append(float(np.mean(class_scores)))
    return float(np.mean(tolerance_scores))

def evaluate(model, loader):
    model.eval()
    true_class, true_azimuth, pred_class, pred_azimuth = [], [], [], []
    with torch.inference_mode():
        for batch in loader:
            sound_logits, azimuth_logits = model(batch["features"].to(DEVICE))
            true_class.extend(batch["class_idx"].tolist())
            true_azimuth.extend(batch["azimuth_idx"].tolist())
            pred_class.extend(sound_logits.argmax(1).cpu().tolist())
            pred_azimuth.extend(azimuth_logits.argmax(1).cpu().tolist())
    return joint_macro_f1(
        [SOUND_CLASSES[i] for i in true_class],
        [AZIMUTHS[i] for i in true_azimuth],
        [SOUND_CLASSES[i] for i in pred_class],
        [AZIMUTHS[i] for i in pred_azimuth],
    )

model = SpatialBaselineCNN(
    num_classes=len(SOUND_CLASSES), num_azimuths=len(AZIMUTHS)
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
checkpoint = WORK / "best_model.pt"
epochs = 1 if FAST_DEV_RUN else 16
best_score = -1.0

for epoch in range(1, epochs + 1):
    model.train()
    loss_sum = 0.0
    count = 0
    for batch in train_loader:
        features = batch["features"].to(DEVICE, non_blocking=True)
        class_target = batch["class_idx"].to(DEVICE, non_blocking=True)
        azimuth_target = batch["azimuth_idx"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        sound_logits, azimuth_logits = model(features)
        loss = criterion(sound_logits, class_target) + criterion(azimuth_logits, azimuth_target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        loss_sum += float(loss.detach()) * len(features)
        count += len(features)
    score = evaluate(model, validation_loader)
    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), checkpoint)
    print(f"epoch={epoch:02d} loss={loss_sum / count:.4f} joint={score:.4f}")

model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
print("best validation Joint Macro-F1:", round(best_score, 4))


## 추론과 제출

기본 실행은 전체 test를 예측해 `/kaggle/working/submission.csv`를 만듭니다.


In [ ]:
predicted_classes, predicted_azimuths = [], []
model.eval()
with torch.inference_mode():
    for batch in test_loader:
        sound_logits, azimuth_logits = model(batch["features"].to(DEVICE))
        predicted_classes.extend(
            SOUND_CLASSES[index] for index in sound_logits.argmax(1).cpu().tolist()
        )
        predicted_azimuths.extend(
            AZIMUTHS[index] for index in azimuth_logits.argmax(1).cpu().tolist()
        )

submission = pd.DataFrame({
    "id": test_run["id"].astype(str).tolist(),
    "sound_class": predicted_classes,
    "azimuth": predicted_azimuths,
})
expected_sample = sample.set_index(sample["id"].astype(str)).loc[
    submission["id"].astype(str)
].reset_index(drop=True)
assert submission.columns.tolist() == ["id", "sound_class", "azimuth"]
assert submission["id"].tolist() == expected_sample["id"].astype(str).tolist()
assert submission["id"].is_unique and not submission.isna().any().any()
assert set(submission["sound_class"]).issubset(SOUND_CLASSES)
assert set(submission["azimuth"]).issubset(AZIMUTHS)

if FAST_DEV_RUN:
    output_path = WORK / "fast_dev_predictions.csv"
    print("FAST_DEV_RUN 결과는 제출용이 아닙니다.")
else:
    assert len(submission) == len(test) == len(sample)
    output_path = WORK / "submission.csv"
submission.to_csv(output_path, index=False)
print("saved:", output_path, "rows:", len(submission))
print(submission.head())

for extracted in (extracted_train, extracted_test):
    if extracted is not None and extracted.exists() and extracted.is_relative_to(WORK):
        shutil.rmtree(extracted)
